In [ ]:
# /// script
# requires-python = ">=3.12"
# dependencies = []
# ///

In [ ]:
from glob import glob
from datetime import datetime, timedelta

import pandas as pd
import numpy as np

from scipy.optimize import curve_fit
import cv2

from functools import partial

import matplotlib.pyplot as plt
from sqlalchemy import create_engine

In [ ]:
mysql_username = "root"
mysql_password = "gctk4321"
mysql_ip = "192.168.0.101"
port = "13306"
db = "data"

engine = create_engine('mysql+pymysql://{}:{}@{}:{}/{}'.format(mysql_username, mysql_password, mysql_ip, port, db))

In [ ]:
def relocate(position, array):
    """
    按预设位置对分层参数进行加权平均（匹配实际大气分层位置）
    
    参数:
        array (list/np.ndarray): 待平均的分层参数数组（长度应等于layers）
        
    返回:
        np.ndarray: 平均后的参数数组（长度等于layers）
    """
    assert len(array) == len(position)
    
    ave = np.zeros_like(array)
    ave[0] = ((array[0] + array[1] + array[2] + array[3] + array[4]) * 20 + array[5] * 100) / 200
    # 后续层线性插值（根据position数组的位置间隔）
    for i in range(15):
        d = (i+2) * 200  # 当前层的位置（间隔200米）
        for j in range(15):
            # 找到d所在的position区间，进行线性插值
            if position[j] <= d <= position[j+1]:
                ave[i+1] = ((d - position[j]) * array[j] + (position[j+1] - d) * array[j+1]) / (position[j+1] - position[j])
            else:
                ave[i+1] = array[-1]  # 超出范围时取最后一个值
    return ave

In [ ]:
target_time ='''2025/6/6	10:24
2025/6/6	10:36
2025/6/6	10:45
2025/6/6	10:55
2025/6/6	11:26
2025/6/6	11:37
2025/6/6	14:29
2025/6/6	14:49
2025/6/9	11:42
2025/6/9	11:47
2025/6/9	13:26
2025/6/9	13:36
2025/6/9	13:46
2025/6/9	15:09
2025/6/9	15:18
2025/6/10	13:45
2025/6/10	13:52
2025/6/10	14:57
2025/6/10	15:06
2025/6/10	15:56
2025/6/10	16:02
2025/6/11	16:35
2025/6/11	16:47
2025/6/11	16:58
2025/6/12	9:04
2025/6/12	9:13
2025/6/12	9:17
2025/6/12	9:39
2025/6/12	9:42
2025/6/12	11:29
2025/6/12	11:35'''.split('\n')

target_time_df = pd.DataFrame(target_time, columns=['time'])
target_time_df['time'] = pd.to_datetime(target_time_df['time'], format="%Y/%m/%d\t%H:%M")

target_time_df

In [ ]:
wind_df = pd.read_excel(r'data\20250611\大气数据\20250611_atmosphere\Wind\Wind.xlsx')
wind_df['time'] = pd.to_datetime('2025-'+wind_df['m/s'])
wind_df.drop('m/s', axis=1, inplace=True)
wind_df.info()

position = [int(p[:-1]) for p in wind_df.columns if p[:-1].isdigit()]
ave_at_pos = partial(relocate, position)
def ave(array):
    return np.mean(ave_at_pos(array))

wind_df['avg'] = wind_df[[p for p in wind_df.columns if p[:-1].isdigit()]].apply(ave, axis=1)
wind_df['avg'].plot()

In [ ]:
temporature_df = pd.read_excel(r'data\20250611\大气数据\20250611_atmosphere\Temperature\Temperature.xlsx')
temporature_df['time'] = pd.to_datetime('2025-'+temporature_df['°C'])
temporature_df.drop('°C', axis=1, inplace=True)
temporature_df.info()

position = [int(p[:-1]) for p in temporature_df.columns if p[:-1].isdigit()]
ave_at_pos = partial(relocate, position)
def ave(array):
    return np.mean(ave_at_pos(array))

temporature_df['avg'] = temporature_df[[p for p in temporature_df.columns if p[:-1].isdigit()]].apply(ave, axis=1)
temporature_df['avg'].plot()